# 06 · 송도 TTC·PET 단변량·이변량 EVT

05의 사건 목록으로 M1·M2의 극값모형을 적합합니다. **사고자료 대조가 없으므로 이 노트북은 모형의 정확도를 판정하지 않습니다.** 같은 추정 대상에서 단변량·이변량 결과가 얼마나 다른지와 그 불확실성을 보여줍니다(AGENTS.md, 08 §7).

| 단계 | 방법 | 근거 |
|---|---|---|
| 입력 | `X = −minTTC`, `Y = −PET`(후미는 `−minPET`). 역수 변환 없음 | M1 §3.3, 사용자 규칙 |
| 층(strata) | 후미추돌(M2 계열)과 교차 **좌회전×대향직진**(M1 계열)을 따로. 지점별 + 전 지점 합산(보조) | 11·12, 두 유형 혼합 금지 |
| 후미 대상 차로 | 기본: 04의 `through_or_shared` 차로(M2 p.3 회전 전용차로 제외). 감도: 접근로 전 차로 | 04 결과: 전용·공용 경계가 뚜렷하지 않음 |
| 임계값 | 기본: 상위 12% 분위(M2 p.5, DuMouchel 1983·Sun 2013). 평균잔차수명·모수 안정성 그림으로 확인하고 필요하면 층별로 바꿈(M1 p.7) | M1·M2 |
| 표본 충분성 | 초과 표본 30개 미만이면 `표본부족` 표시 | M2 p.5 "보통 30개 이상" |
| 단변량 | 일반화 파레토(GPD) 최대우도 | M1 식(2) |
| 이변량 | 로지스틱 이변량 GP, 한쪽만 초과한 관측을 쓰는 **검열우도** | M1 식(7), M2 §4.1 식(11)–(12), Ledford & Tawn 1996 |
| 0초 경계 도달 | 단변량 `Pr(Z≥0)`, 이변량 `Pr(X≥0 또는 Y≥0) = 1 − F(0,0)` | M1 식(9)–(10), M2 §4.2 |
| 불확실성 | 추정 모수의 점근 정규분포에서 5,000번 모의추출한 95% 구간 | M1·M2의 모의추출 절차 |

**같은 대상끼리 비교합니다(08 §7).** 단변량 TTC는 "TTC가 0초에 닿는 사건", 단변량 PET는 "PET가 0초에 닿는 사건", 이변량은 "둘 중 하나라도 닿는 사건"을 추정하므로 서로 다른 값입니다. 그래서 이변량 모형의 **주변(marginal) 확률**을 단변량과 나란히 두고, 두 단변량을 **독립으로 결합한 기준값**을 이변량 OR 확률과 나란히 둡니다. 관측시간당(시간당) 값으로도 나타내며, 관측시간은 04에서 차량이 찍힌 프레임 수로 잰 값입니다.

0초를 넘는 부분은 **사고 쪽으로의 외삽**이지 관측된 사고가 아닙니다. 연(年) 단위 환산은 관측 구간이 1년을 대표한다는 가정이 필요하므로 하지 않습니다.

## 1. 준비

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.util
import io
import json
import math
import time
import uuid
import warnings

packages = {"pandas": "pandas", "numpy": "numpy", "scipy": "scipy", "matplotlib": "matplotlib", "IPython": "IPython"}
missing = [p for m, p in packages.items() if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError("별도 셀에서 설치하세요: %pip install " + " ".join(missing))
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
print("준비 완료 | pandas", pd.__version__, "| numpy", np.__version__)

## 2. 입력 실행·설정
- **`SOURCE05_ID`에 05 마지막 셀의 실행 ID를 넣으세요.** 04 실행 ID는 채워 두었습니다.
- `UPPER_SHARE = 0.12`: 임계값 기본값(상위 12%). 층별로 바꾸려면 `THRESHOLD_OVERRIDE`에 `{"층 이름": {"ttc": u, "pet": u}}` 형식으로 넣습니다. 값은 음수화한 지표 기준입니다(예: TTC 1.5초 → −1.5).

In [ ]:
SOURCE04_ID = "20260922T182837Z_4eaba3c7"
SOURCE05_ID = ""  # 05 마지막 셀의 실행 ID
if not SOURCE05_ID:
    raise ValueError("05 노트북 마지막 셀의 실행 ID를 SOURCE05_ID에 입력하세요.")
PROJECT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                if (p / "AGENTS.md").is_file() and (p / "data/raw").is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError("프로젝트 또는 analysis 폴더에서 실행하세요.")
SOURCE04 = (PROJECT / "data/processed/songdo_structure" / SOURCE04_ID).resolve()
SOURCE05 = (PROJECT / "data/processed/songdo_events" / SOURCE05_ID).resolve()
OUTPUT_ROOT = (PROJECT / "data/processed/songdo_evt").resolve()
FPS = 29.97
UPPER_SHARE = 0.12
MIN_EXCEEDANCES = 30
N_SIMULATIONS = 5000
RANDOM_SEED = 20260923
THRESHOLD_OVERRIDE = {}
SITES = list("ABCEFGHIJKLMNOPQRSTU")
print("입력 04:", SOURCE04_ID, "| 입력 05:", SOURCE05_ID)

## 3. 입력 읽기와 관측시간
- 05가 완료된 실행인지 확인하고, 두 사건 목록에서 EVT에 필요한 열만 읽습니다.
- **관측시간**: 04의 드론별 `n_unique_times`(차량이 찍힌 서로 다른 프레임 수) ÷ 29.97 ÷ 3600. 05가 드론 간 동시 관측을 발견했다면 같은 장면이 두 번 세어질 수 있으므로 경고합니다.

In [ ]:
def now_utc():
    return datetime.now(timezone.utc).isoformat()


def target_path(path):
    path = Path(path).resolve()
    if not path.is_relative_to(RUN_DIR.resolve()):
        raise ValueError("이번 실행 폴더 밖에는 저장하지 않습니다.")
    return path


def save_csv(table, path):
    target = target_path(path)
    temporary = target.with_name(target.name + ".tmp")
    table.to_csv(temporary, index=False, encoding="utf-8-sig")
    temporary.replace(target)


def save_json(value, path):
    target = target_path(path)
    temporary = target.with_name(target.name + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    temporary.replace(target)


meta05 = json.loads((SOURCE05 / "run_metadata.json").read_text(encoding="utf-8"))
if meta05.get("status") != "completed" or meta05.get("source04_run_id") != SOURCE04_ID:
    raise ValueError("05가 완료된 실행이 아니거나 04 출처가 다릅니다: " + str(meta05.get("status")))
rear_cols = ["site", "file_stem", "drone_id", "section", "lane", "lane_role", "lane_through_share",
             "leader_id", "follower_id", "ttc_status", "min_ttc_s", "pet_status", "min_pet_s"]
cross_cols = ["site", "file_stem", "drone_id", "vehicle_a_id", "vehicle_b_id", "crossing_class",
              "m1_left_turn_opposing_through", "ttc_status", "min_ttc_s", "pet_status", "pet_s"]
rear = pd.read_csv(SOURCE05 / "event_catalog_rear_end.csv", usecols=rear_cols,
                   dtype={"section": "string", "lane": "string", "drone_id": "string"})
cross = pd.read_csv(SOURCE05 / "event_catalog_crossing.csv", usecols=cross_cols, dtype={"drone_id": "string"})
progress05 = pd.read_csv(SOURCE05 / "progress.csv", keep_default_na=False)
simultaneous = pd.to_numeric(progress05.get("max_drone_simultaneous_s", pd.Series(dtype=float)), errors="coerce").fillna(0)
if (simultaneous > 0).any():
    print("주의: 드론 간 동시 관측이 있는 파일", int((simultaneous > 0).sum()), "개 — 같은 장면이 두 번 세어질 수 있습니다.")

drones = pd.read_csv(SOURCE04 / "time_structure_by_drone.csv")
hours = (drones.groupby("site")["n_unique_times"].sum() / FPS / 3600).to_dict()
hours["ALL"] = float(sum(hours.values()))
print("사건: 후미추돌", len(rear), "| 교차", len(cross), "| 관측시간 합계", round(hours["ALL"], 1), "시간")

## 4. 층(strata) 만들기
각 층은 한 유형·한 지점(또는 전 지점 합산)의 사건입니다. 두 유형과 서로 다른 PET 정의를 한 층에 섞지 않습니다.
- `후미_기본`: 후미추돌, `lane_role = through_or_shared`
- `후미_전차로`: 후미추돌, 접근로 전 차로(감도분석)
- `교차_M1`: 교차 중 좌회전×대향직진

In [ ]:
def as_negated(table, ttc_col, pet_col):
    x = np.where(table["ttc_status"].eq("computed"), -pd.to_numeric(table[ttc_col], errors="coerce"), np.nan)
    y = np.where(table["pet_status"].eq("computed"), -pd.to_numeric(table[pet_col], errors="coerce"), np.nan)
    return pd.DataFrame({"site": table["site"].to_numpy(), "x": x, "y": y})


kinds = {
    "후미_기본": as_negated(rear.loc[rear["lane_role"].eq("through_or_shared")], "min_ttc_s", "min_pet_s"),
    "후미_전차로": as_negated(rear, "min_ttc_s", "min_pet_s"),
    "교차_M1": as_negated(cross.loc[cross["m1_left_turn_opposing_through"].astype(str).str.lower().eq("true")],
                          "min_ttc_s", "pet_s"),
}
strata = {}
for kind, table in kinds.items():
    strata[(kind, "ALL")] = table
    for site in SITES:
        part = table.loc[table["site"].eq(site)]
        if len(part):
            strata[(kind, site)] = part
summary_rows = [{"유형": k, "지점": s, "사건": len(t), "TTC있음": int(np.isfinite(t["x"]).sum()),
                 "PET있음": int(np.isfinite(t["y"]).sum()),
                 "둘다": int((np.isfinite(t["x"]) & np.isfinite(t["y"])).sum())} for (k, s), t in strata.items()]
strata_table = pd.DataFrame(summary_rows)
display(strata_table.loc[strata_table["지점"].eq("ALL")])

## 5. 단변량 GPD 함수
- 초과량 `z = 값 − u`에 GPD(σ, ξ)를 최대우도로 맞춥니다. ξ ≤ −1에서는 최대우도가 정의되지 않으므로(Smith 1985) ξ > −1로 제한합니다. 표준오차는 수치 헤시안으로 구하며, 추정값이 이 경계에 닿거나 헤시안이 불안정하면 `ok_no_covariance`로 표시합니다.
- 0초 경계 도달: 초과 사건 중 `Pr(Z ≥ 0 | Z > u) = (1 + ξ(0 − u)/σ)^(−1/ξ)`(ξ<0이고 0이 분포의 끝점보다 크면 0). **관측 기간의 기대 건수 = 초과 수 × 이 확률**이며, 시간당 값은 이것을 관측시간으로 나눈 값입니다. M1 식(9)의 R은 조건부 확률 그 자체입니다.
- 평균잔차수명 그림과 모수 안정성 그림(수정 척도 σ* = σ − ξu, 형상 ξ)을 그려 임계값을 확인합니다(M1 p.7).

In [ ]:
def gpd_nll(params, z):
    log_sigma, xi = params
    if xi <= -1.0:
        return np.inf
    sigma = math.exp(log_sigma)
    if abs(xi) < 1e-8:
        return len(z) * log_sigma + z.sum() / sigma
    t = 1.0 + xi * z / sigma
    if np.any(t <= 0):
        return np.inf
    return len(z) * log_sigma + (1.0 + 1.0 / xi) * np.log(t).sum()


def numerical_hessian(f, x, rel=1e-4):
    x = np.asarray(x, dtype=float)
    n = len(x)
    h = rel * np.maximum(1.0, np.abs(x))
    hess = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            ei = np.zeros(n); ei[i] = h[i]
            ej = np.zeros(n); ej[j] = h[j]
            value = (f(x + ei + ej) - f(x + ei - ej) - f(x - ei + ej) + f(x - ei - ej)) / (4 * h[i] * h[j])
            hess[i, j] = hess[j, i] = value
    return hess


def safe_covariance(hess):
    try:
        cov = np.linalg.inv(hess)
        if np.all(np.isfinite(cov)) and np.all(np.linalg.eigvalsh((cov + cov.T) / 2) > 0):
            return (cov + cov.T) / 2
    except np.linalg.LinAlgError:
        pass
    return None


def fit_gpd(z):
    start = np.array([math.log(max(z.mean(), 1e-6)), 0.1])
    best = None
    for xi0 in (0.1, -0.2, 0.3):
        start[1] = xi0
        result = minimize(gpd_nll, start, args=(z,), method="Nelder-Mead",
                          options={"xatol": 1e-8, "fatol": 1e-10, "maxiter": 4000})
        if np.isfinite(result.fun) and (best is None or result.fun < best.fun):
            best = result
    cov = safe_covariance(numerical_hessian(lambda p: gpd_nll(p, z), best.x))
    return best.x, cov, float(best.fun)


def conditional_boundary_prob(u, sigma, xi, level=0.0):
    z = level - u
    if abs(xi) < 1e-8:
        return math.exp(-z / sigma)
    t = 1.0 + xi * z / sigma
    return 0.0 if t <= 0 else t ** (-1.0 / xi)


def univariate(values, u):
    values = values[np.isfinite(values)]
    exceed = values[values > u] - u
    out = {"n": int(len(values)), "threshold": float(u), "n_exceed": int(len(exceed)),
           "enough_exceedances": bool(len(exceed) >= MIN_EXCEEDANCES)}
    if len(exceed) < 10:
        return {**out, "fit": "too_few"}
    params, cov, nll = fit_gpd(exceed)
    sigma, xi = math.exp(params[0]), params[1]
    p_cond = conditional_boundary_prob(u, sigma, xi)
    out.update(fit="ok", sigma=sigma, xi=xi, nll=nll, p_boundary_given_exceed=p_cond,
               expected_boundary_events=len(exceed) * p_cond)
    if cov is not None:
        se = np.sqrt(np.diag(cov))
        out.update(se_log_sigma=float(se[0]), se_xi=float(se[1]))
        draws = rng.multivariate_normal(params, cov, size=N_SIMULATIONS)
        sims = np.array([len(exceed) * conditional_boundary_prob(u, math.exp(a), b) for a, b in draws])
        out.update(expected_low=float(np.quantile(sims, 0.025)), expected_high=float(np.quantile(sims, 0.975)))
    else:
        out.update(fit="ok_no_covariance")
    return out


def threshold_diagnostics(values, title, path):
    values = np.sort(values[np.isfinite(values)])
    if len(values) < 50:
        return
    grid = np.quantile(values, np.linspace(0.50, 0.98, 25))
    mrl, mrl_ci, scale_star, shape, shape_ci = [], [], [], [], []
    for u in grid:
        z = values[values > u] - u
        mrl.append(z.mean()); mrl_ci.append(1.96 * z.std(ddof=1) / math.sqrt(len(z)))
        if len(z) >= 10:
            params, cov, _ = fit_gpd(z)
            sigma, xi = math.exp(params[0]), params[1]
            scale_star.append(sigma - xi * u); shape.append(xi)
            shape_ci.append(1.96 * math.sqrt(cov[1, 1]) if cov is not None else np.nan)
        else:
            scale_star.append(np.nan); shape.append(np.nan); shape_ci.append(np.nan)
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
    axes[0].errorbar(grid, mrl, yerr=mrl_ci, fmt="o-", ms=3); axes[0].set_title("평균잔차수명")
    axes[1].plot(grid, scale_star, "o-", ms=3); axes[1].set_title("수정 척도 σ*")
    axes[2].errorbar(grid, shape, yerr=shape_ci, fmt="o-", ms=3); axes[2].set_title("형상 ξ")
    u_default = np.quantile(values, 1 - UPPER_SHARE)
    for ax in axes:
        ax.axvline(u_default, color="red", lw=1, ls="--")
        ax.set_xlabel("임계값 u (음수화 지표, 초)")
    fig.suptitle(title + " | 빨간 선 = 상위 12% 기본 임계값")
    fig.tight_layout()
    fig.savefig(target_path(path), dpi=110)
    plt.close(fig)

print("단변량 함수 정의 완료")

## 6. 이변량 로지스틱 GP 함수 (검열우도)
M2 §4.1과 Coles(2001) §8.3을 따릅니다.
- 주변분포: 각 지표의 GPD 꼬리와 초과 비율 ζ로 `F(v) = 1 − ζ(1 + ξ(v − u)/σ)^(−1/ξ)`, 이를 단위 Fréchet `z = −1/log F`로 바꿉니다.
- 결합: `G(z₁, z₂) = exp{−(z₁^(−1/α) + z₂^(−1/α))^α}`, α∈(0,1]. α→1은 독립, α→0은 완전 종속입니다.
- 네 영역: 둘 다 임계값 이하(`G(u)`), 한쪽만 초과(`∂G/∂x` 또는 `∂G/∂y`), 둘 다 초과(`∂²G/∂x∂y`).
- 경계: `Pr(X≥0 또는 Y≥0) = 1 − G(z₁(0), z₂(0))`. 같은 모형의 주변 확률 `Pr(X≥0)`, `Pr(Y≥0)`와 독립 결합 `1 − (1 − p_x)(1 − p_y)`도 계산합니다.

In [ ]:
def margin_terms(values, u, sigma, xi, zeta):
    exceed = values > u
    z = np.full(len(values), -1.0 / math.log1p(-zeta))
    log_dz = np.zeros(len(values))
    if exceed.any():
        w = values[exceed] - u
        if abs(xi) < 1e-8:
            tail = zeta * np.exp(-w / sigma)
            dens = zeta / sigma * np.exp(-w / sigma)
        else:
            t = 1.0 + xi * w / sigma
            if np.any(t <= 0):
                return None
            tail = zeta * t ** (-1.0 / xi)
            dens = zeta / sigma * t ** (-1.0 / xi - 1.0)
        if np.any(tail >= 1) or np.any(tail <= 0):
            return None
        log_f = np.log1p(-tail)
        z_ex = -1.0 / log_f
        z[exceed] = z_ex
        log_dz[exceed] = 2.0 * np.log(z_ex) + np.log(dens) - log_f
    return z, log_dz, exceed


def unpack(theta):
    return (math.exp(theta[0]), theta[1], math.exp(theta[2]), theta[3], 1.0 / (1.0 + math.exp(-theta[4])))


def bgp_nll(theta, x, y, ux, uy, zeta_x, zeta_y):
    sx, xx, sy, xy, a = unpack(theta)
    if xx <= -1.0 or xy <= -1.0:
        return np.inf
    mx = margin_terms(x, ux, sx, xx, zeta_x)
    my = margin_terms(y, uy, sy, xy, zeta_y)
    if mx is None or my is None:
        return np.inf
    z1, ld1, e1 = mx
    z2, ld2, e2 = my
    p1, p2 = z1 ** (-1.0 / a), z2 ** (-1.0 / a)
    s = p1 + p2
    v = s ** a
    v1 = -(p1 / z1) * s ** (a - 1.0)
    v2 = -(p2 / z2) * s ** (a - 1.0)
    v12 = ((a - 1.0) / a) * (p1 / z1) * (p2 / z2) * s ** (a - 2.0)
    ll = -v
    r10, r01, r11 = e1 & ~e2, ~e1 & e2, e1 & e2
    with np.errstate(divide="ignore", invalid="ignore"):
        ll = ll + np.where(r10, np.log(-v1) + ld1, 0.0)
        ll = ll + np.where(r01, np.log(-v2) + ld2, 0.0)
        ll = ll + np.where(r11, np.log(v1 * v2 - v12) + ld1 + ld2, 0.0)
    total = ll.sum()
    return -total if np.isfinite(total) else np.inf


def boundary_z(u, sigma, xi, zeta):
    tail = zeta * conditional_boundary_prob(u, sigma, xi)
    if tail <= 0:
        return np.inf, 0.0
    if tail >= 1:
        return 0.0, 1.0
    return -1.0 / math.log1p(-tail), tail


def bgp_boundary(theta, ux, uy, zeta_x, zeta_y):
    sx, xx, sy, xy, a = unpack(theta)
    z1, p_x = boundary_z(ux, sx, xx, zeta_x)
    z2, p_y = boundary_z(uy, sy, xy, zeta_y)
    if z1 == 0.0 or z2 == 0.0:
        return 1.0, p_x, p_y, 1.0 - (1.0 - p_x) * (1.0 - p_y)
    part = (0.0 if np.isinf(z1) else z1 ** (-1.0 / a)) + (0.0 if np.isinf(z2) else z2 ** (-1.0 / a))
    p_or = -math.expm1(-(part ** a)) if part > 0 else 0.0
    return p_or, p_x, p_y, 1.0 - (1.0 - p_x) * (1.0 - p_y)


def bivariate(pairs, ux, uy, start_x, start_y):
    x, y = pairs["x"].to_numpy(float), pairs["y"].to_numpy(float)
    zeta_x, zeta_y = float(np.mean(x > ux)), float(np.mean(y > uy))
    out = {"n_pairs": int(len(x)), "n_exceed_x": int((x > ux).sum()), "n_exceed_y": int((y > uy).sum()),
           "n_exceed_both": int(((x > ux) & (y > uy)).sum()), "zeta_x": zeta_x, "zeta_y": zeta_y}
    if min(out["n_exceed_x"], out["n_exceed_y"]) < 10 or zeta_x >= 1 or zeta_y >= 1:
        return {**out, "fit": "too_few"}
    theta0 = np.array([math.log(start_x[0]), start_x[1], math.log(start_y[0]), start_y[1], 2.0])
    best = None
    for a0 in (2.0, 0.0, -1.0):
        theta0[4] = a0
        result = minimize(bgp_nll, theta0, args=(x, y, ux, uy, zeta_x, zeta_y), method="Nelder-Mead",
                          options={"xatol": 1e-7, "fatol": 1e-9, "maxiter": 8000})
        if np.isfinite(result.fun) and (best is None or result.fun < best.fun):
            best = result
    if best is None:
        return {**out, "fit": "failed"}
    sx, xx, sy, xy, a = unpack(best.x)
    p_or, p_x, p_y, p_ind = bgp_boundary(best.x, ux, uy, zeta_x, zeta_y)
    n = len(x)
    out.update(fit="ok", sigma_x=sx, xi_x=xx, sigma_y=sy, xi_y=xy, alpha=a, nll=float(best.fun),
               expected_or=n * p_or, expected_x=n * p_x, expected_y=n * p_y, expected_or_independent=n * p_ind)
    cov = safe_covariance(numerical_hessian(lambda t: bgp_nll(t, x, y, ux, uy, zeta_x, zeta_y), best.x))
    if cov is None:
        out["fit"] = "ok_no_covariance"
        return out
    draws = rng.multivariate_normal(best.x, cov, size=N_SIMULATIONS)
    sims = np.array([bgp_boundary(d, ux, uy, zeta_x, zeta_y) for d in draws]) * n
    for k, name in enumerate(["or", "x", "y", "or_independent"]):
        out[f"expected_{name}_low"] = float(np.quantile(sims[:, k], 0.025))
        out[f"expected_{name}_high"] = float(np.quantile(sims[:, k], 0.975))
    alpha_draws = 1.0 / (1.0 + np.exp(-draws[:, 4]))
    out.update(alpha_low=float(np.quantile(alpha_draws, 0.025)), alpha_high=float(np.quantile(alpha_draws, 0.975)))
    return out

print("이변량 함수 정의 완료")

## 7. 이번 실행 폴더

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = (OUTPUT_ROOT / RUN_ID).resolve()
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR / "diagnostics").mkdir()
rng = np.random.default_rng(RANDOM_SEED)
run_metadata = {"run_id": RUN_ID, "source04_run_id": SOURCE04_ID, "source05_run_id": SOURCE05_ID,
                "started_at_utc": now_utc(), "status": "running", "upper_share": UPPER_SHARE,
                "min_exceedances": MIN_EXCEEDANCES, "n_simulations": N_SIMULATIONS, "seed": RANDOM_SEED,
                "threshold_override": THRESHOLD_OVERRIDE, "crash_data_validation": "not_available",
                "negation": "X=-minTTC, Y=-PET (rear: -minPET); no inverse transform"}
save_json(run_metadata, RUN_DIR / "run_metadata.json")
save_csv(strata_table, RUN_DIR / "strata_sizes.csv")
print("이번 결과 폴더:", RUN_DIR)

## 8. 단변량 적합과 임계값 진단
층마다 TTC·PET 각각의 임계값을 정하고 GPD를 적합합니다. 전 지점 합산 층의 진단 그림을 아래에 보여주고, 모든 층의 그림은 `diagnostics/`에 저장합니다. **그림에서 빨간 선(기본 임계값) 근처가 평균잔차수명이 거의 직선이고 σ*·ξ가 안정적인 구간인지 확인**하세요. 아니라면 `THRESHOLD_OVERRIDE`로 바꾸고 7번부터 다시 실행합니다.

In [ ]:
started = time.perf_counter()
uni_rows, thresholds = [], {}
for (kind, site), table in strata.items():
    name = f"{kind}|{site}"
    for label, column in [("ttc", "x"), ("pet", "y")]:
        values = table[column].to_numpy(float)
        finite = values[np.isfinite(values)]
        if len(finite) < 10:
            uni_rows.append({"유형": kind, "지점": site, "지표": label, "n": len(finite), "fit": "too_few"})
            continue
        u = THRESHOLD_OVERRIDE.get(name, {}).get(label, float(np.quantile(finite, 1 - UPPER_SHARE)))
        thresholds[(kind, site, label)] = u
        result = univariate(values, u)
        uni_rows.append({"유형": kind, "지점": site, "지표": label, "hours": hours.get(site, np.nan), **result})
        threshold_diagnostics(values, f"{kind} {site} {label.upper()}",
                              RUN_DIR / "diagnostics" / f"{kind}_{site}_{label}.png")
univariate_table = pd.DataFrame(uni_rows)
for col in ["expected_boundary_events", "expected_low", "expected_high"]:
    if col in univariate_table:
        univariate_table[col.replace("expected", "per_hour")] = univariate_table[col] / univariate_table["hours"]
save_csv(univariate_table, RUN_DIR / "univariate_gpd.csv")
print(f"단변량 적합 완료 ({time.perf_counter() - started:.0f}초)")
show = ["유형", "지점", "지표", "n", "threshold", "n_exceed", "enough_exceedances", "sigma", "xi", "se_xi",
        "p_boundary_given_exceed", "per_hour_boundary_events", "per_hour_low", "per_hour_high"]
display(univariate_table.loc[univariate_table["지점"].eq("ALL"), [c for c in show if c in univariate_table]].round(5))
from IPython.display import Image as _Image
for kind in kinds:
    for label in ["ttc", "pet"]:
        path = RUN_DIR / "diagnostics" / f"{kind}_ALL_{label}.png"
        if path.exists():
            display(_Image(filename=str(path)))

## 9. 이변량 적합
TTC와 PET가 **모두 계산된 사건**만 씁니다. 이 조건 때문에 분석 모집단이 단변량보다 좁아지며(08 §5), 그 수를 함께 표시합니다. 임계값은 같은 층의 단변량 임계값을 씁니다(M1 §5.2).

In [ ]:
started = time.perf_counter()
bi_rows = []
uni_lookup = univariate_table.set_index(["유형", "지점", "지표"])
for (kind, site), table in strata.items():
    pairs = table.loc[np.isfinite(table["x"]) & np.isfinite(table["y"])]
    key_x, key_y = (kind, site, "ttc"), (kind, site, "pet")
    if key_x not in thresholds or key_y not in thresholds or len(pairs) < 20:
        bi_rows.append({"유형": kind, "지점": site, "n_pairs": len(pairs), "fit": "too_few"})
        continue
    ux, uy = thresholds[key_x], thresholds[key_y]
    rx, ry = uni_lookup.loc[key_x], uni_lookup.loc[key_y]
    start_x = (float(rx.get("sigma", 0.5)) if pd.notna(rx.get("sigma", np.nan)) else 0.5,
               float(rx.get("xi", 0.0)) if pd.notna(rx.get("xi", np.nan)) else 0.0)
    start_y = (float(ry.get("sigma", 0.5)) if pd.notna(ry.get("sigma", np.nan)) else 0.5,
               float(ry.get("xi", 0.0)) if pd.notna(ry.get("xi", np.nan)) else 0.0)
    result = bivariate(pairs, ux, uy, start_x, start_y)
    bi_rows.append({"유형": kind, "지점": site, "hours": hours.get(site, np.nan), "threshold_x": ux,
                    "threshold_y": uy, **result})
bivariate_table = pd.DataFrame(bi_rows)
for col in [c for c in bivariate_table.columns if c.startswith("expected_")]:
    bivariate_table[col.replace("expected_", "per_hour_")] = bivariate_table[col] / bivariate_table["hours"]
save_csv(bivariate_table, RUN_DIR / "bivariate_logistic_gp.csv")
print(f"이변량 적합 완료 ({time.perf_counter() - started:.0f}초)")
show = ["유형", "지점", "n_pairs", "n_exceed_x", "n_exceed_y", "n_exceed_both", "alpha", "alpha_low", "alpha_high",
        "per_hour_or", "per_hour_or_low", "per_hour_or_high", "per_hour_or_independent"]
display(bivariate_table.loc[bivariate_table["지점"].eq("ALL"), [c for c in show if c in bivariate_table]].round(5))

## 10. 같은 대상끼리 나란히 비교
층마다 시간당 "0초 경계에 닿는 사건"의 추정값을 나란히 둡니다.
- `단변량_TTC` vs `이변량_주변_TTC`: 같은 대상(TTC가 0초에 닿음)을 단변량 모형과 이변량 모형이 각각 추정한 값. 단, 이변량은 두 지표가 모두 있는 사건만 쓰므로 모집단이 다릅니다.
- `이변량_OR` vs `독립결합_OR`: 같은 대상(둘 중 하나라도 0초)을, 의존성을 추정한 모형과 독립을 가정한 모형이 각각 추정한 값. 두 값의 차이가 **의존구조 모델링이 더하는 부분**입니다.
- 구간(95%)이 넓으면 그 층의 결과는 결론을 뒷받침하기 어렵습니다. **사고자료가 없으므로 어느 쪽이 더 정확한지는 판정하지 않습니다.**

In [ ]:
rows = []
for (kind, site) in strata:
    uni_t = uni_lookup.loc[(kind, site, "ttc")] if (kind, site, "ttc") in uni_lookup.index else None
    uni_p = uni_lookup.loc[(kind, site, "pet")] if (kind, site, "pet") in uni_lookup.index else None
    bi = bivariate_table.loc[bivariate_table["유형"].eq(kind) & bivariate_table["지점"].eq(site)]
    bi = bi.iloc[0] if len(bi) else None
    h = hours.get(site, np.nan)

    def per_hour(row, col):
        return float(row[col]) / h if row is not None and col in row and pd.notna(row[col]) else np.nan

    rows.append({"유형": kind, "지점": site,
                 "단변량_TTC": per_hour(uni_t, "expected_boundary_events"),
                 "단변량_PET": per_hour(uni_p, "expected_boundary_events"),
                 "이변량_주변_TTC": per_hour(bi, "expected_x"), "이변량_주변_PET": per_hour(bi, "expected_y"),
                 "이변량_OR": per_hour(bi, "expected_or"), "독립결합_OR": per_hour(bi, "expected_or_independent"),
                 "이변량_OR_하한": per_hour(bi, "expected_or_low"), "이변량_OR_상한": per_hour(bi, "expected_or_high"),
                 "alpha": float(bi["alpha"]) if bi is not None and "alpha" in bi and pd.notna(bi["alpha"]) else np.nan,
                 "TTC초과충분": bool(uni_t["enough_exceedances"]) if uni_t is not None and pd.notna(uni_t.get("enough_exceedances")) else False,
                 "PET초과충분": bool(uni_p["enough_exceedances"]) if uni_p is not None and pd.notna(uni_p.get("enough_exceedances")) else False})
comparison = pd.DataFrame(rows)
save_csv(comparison, RUN_DIR / "comparison_per_hour.csv")
print("시간당 0초 경계 도달 추정(외삽) — 사고 건수가 아님")
display(comparison.loc[comparison["지점"].eq("ALL")].round(6))
print("지점별 표(초과 표본 충분 여부 포함)는 comparison_per_hour.csv에 있습니다.")
display(comparison.loc[comparison["지점"].ne("ALL") & comparison["유형"].eq("교차_M1")].round(6))

## 11. 저장

In [ ]:
run_metadata.update(status="completed", finished_at_utc=now_utc(),
                    n_strata=len(strata), n_univariate_fits=int(univariate_table["fit"].astype(str).str.startswith("ok").sum()),
                    n_bivariate_fits=int(bivariate_table["fit"].astype(str).str.startswith("ok").sum()))
save_json(run_metadata, RUN_DIR / "run_metadata.json")
print(json.dumps({"실행_ID": RUN_ID, "결과폴더": str(RUN_DIR), "층": len(strata),
                  "단변량적합": run_metadata["n_univariate_fits"], "이변량적합": run_metadata["n_bivariate_fits"],
                  "주의": "사고자료 대조 없음 — 정확도 판정이 아니라 추정 차이·불확실성"}, ensure_ascii=False, indent=2))
print("노트북을 저장하세요.")